# RI-JK 响应计算优化

In [2]:
from pyscf import gto, scf, lib, hessian
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [4]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)

In [6]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()

## HF 响应计算回顾

In [7]:
mo1_rand = np.random.rand(3, nmo, nocc)

In [8]:
np.allclose(
    + 4 * np.einsum("uvP, PQ, klQ, Aqj, kq, lj, up, vi -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
    - np.einsum("uvP, PQ, klQ, Aqj, vq, lj, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
    - np.einsum("uvP, PQ, klQ, Aqj, lq, vj, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
,
    hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand)
)

True

In [9]:
mo1_rand_half_trans = mo_coeff @ mo1_rand
resp_bra_j = 4 * np.einsum("uvP, PQ, klQ, Akj, lj, vi -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
resp_bra_k0 = - 1 * np.einsum("uvP, PQ, klQ, Avj, lj, ki -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
resp_bra_k1 = - 1 * np.einsum("uvP, PQ, klQ, Akj, vj, li -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
r = np.einsum("Aui, up -> Api", (resp_bra_j + resp_bra_k0 + resp_bra_k1), mo_coeff)
np.allclose(r, hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand))

True

## HF 响应优化

### Cholesky 分解积分

首先，我们要真正地开始使用 Cholesky 分解积分，记为 `cderi` $Y_{P, \mu \nu}$。

注意到该积分是具有对称性的，我们一般不将其展开到完整三维张量以避免内存过大损耗。REST 与 PySCF 都使用的是下述 triangular packed (tp) 的形式，只是在 row-major 下三角和 col-major 上三角之间有所不同。Python 这里都是下三角，记为 $Y_{P, \text{tp} (\mu \nu)}$。

In [10]:
cderi_tp = mf.with_df._cderi
cderi_tp.shape

(131, 1225)

对于 K 积分，我们需要将其展开到三维。但实际程序实现时，辅助基上的维度是可以分批的，从而节省一些内存消耗。

In [11]:
cderi = lib.unpack_tril(mf.with_df._cderi)
cderi.shape

(131, 49, 49)

### J 积分响应

J 积分的策略是，充分利用 3c-2e ERI 的对称性，简化一半的内存带宽 (避免不必要的内存解压)。但也需要注意，这种奇淫技巧面对 K 积分时其实没什么卵用，K 积分才是耗时大头。

能省点时间当然稍微省一点，不是坏事。

首先，我们要从半转换的导数分子轨道，得到密度矩阵。

- `mo1_rand`: $U_{p i}^\mathbb{A}$，这是我们求解 CP-KS 方程时的变量。目前的框架下，我们使用的是类似于 withs1 的模式，即第一个轨道是全轨道 (不是通常 CP-KS 方程的非占轨道)，但第二个轨道是占据轨道，可以用来省计算量。
- `mo1_rand_half_trans`: $U_{\mu i}^\mathbb{A} = C_{\mu p} U_{p i}^\mathbb{A}$，这是程序的输入变量 (bra)。

我们将要使用的密度矩阵定义如下：

- `mo1_rand_dm`: $R_{\mu \nu}^\mathbb{A} = U_{\mu i}^\mathbb{A} C_{\nu i} + \text{swap} (\mu, \nu)$

该密度矩阵是对称的。

- `mo1_rand_dm_tp`: $R_{\text{tp} (\mu \nu)}^\mathbb{A} \mathop{\tilde{\bowtie}} R_{\mu \nu}^\mathbb{A}$，即压缩密度矩阵到三角 (row-major 下三角，col-major 上三角)，且将非对角元乘以 2。

In [12]:
mo1_rand_dm = mo1_rand_half_trans @ mocc.T
mo1_rand_dm += mo1_rand_dm.swapaxes(-1, -2)
mo1_rand_dm.shape

(3, 49, 49)

In [13]:
tmp = 2 * mo1_rand_dm
for u in range(nao):
    tmp[:, u, u] *= 0.5
mo1_rand_dm_tp = lib.pack_tril(tmp)
mo1_rand_dm_tp.shape

(3, 1225)

- `itm_j_aux`: $\mathscr{T}_P^\mathbb{A} = Y_{\text{tp} (\mu \nu), P} R_{\text{tp} (\mu \nu)}^\mathbb{A}$

In [14]:
itm_j_aux = mo1_rand_dm_tp @ cderi_tp.T

- `resp_tp_j`: $J_{\text{tp} (\mu \nu)}^\mathbb{A} = 2 \mathscr{T}_P^\mathbb{A} Y_{P, \text{tp} (\mu \nu)}$，这里的系数 2 与最初实现的系数 4 差了两倍；这个两倍来源于 $R_{\mu \nu}^\mathbb{A}$ 的对称化过程。
- `resp_ao_j`: $J_{\mu \nu}^\mathbb{A} \mathop{\tilde{\bowtie}} J_{\text{tp} (\mu \nu)}^\mathbb{A}$，即解压到完整的对称矩阵。
- `resp_bra_j`: $J_{p i}^\mathbb{A} = C_{\mu p} J_{\mu \nu}^\mathbb{A} C_{\nu i}$，即将响应矩阵转换到半转换的导数分子轨道。

In [15]:
resp_tp_j = 2 * itm_j_aux @ cderi_tp
resp_ao_j = lib.unpack_tril(resp_tp_j)
resp_bra_j_recap = resp_ao_j @ mocc
assert np.allclose(resp_bra_j_recap, resp_bra_j)

### K 积分响应 (第一部分)

首先 K 积分需要用的是解压后的 cderi。我们需要对辅助基作分批，在实际程序实现时记得这么搞就好。

首先，我们要生成 3 个中间变量：$Y_{i P \mu}$, $Y_{i P j}$ 与 $\widetilde{Y}_{i P \mu}^\mathbb{A}$。

这里的记号可能比较奇怪。我们之所以将辅助基指标往中间放，只是因为更好地利用多线程乘法优势。

In [16]:
# einsum way
cderi_oxb = np.einsum("Puv, vi -> iPu", cderi, mocc)
# matmul way
cderi_oxb_recap = (mocc.T @ cderi.reshape(naux * nao, nao).T).reshape(nocc, naux, nao)
assert np.allclose(cderi_oxb_recap, cderi_oxb)

In [17]:
# einsum way
cderi_oxo = np.einsum("iPu, uj -> iPj", cderi_oxb, mocc)
# matmul way
cderi_oxo_recap = (cderi_oxb.reshape(nocc * naux, nao) @ mocc).reshape(nocc, naux, nocc)
assert np.allclose(cderi_oxo_recap, cderi_oxo)

In [18]:
# einsum way
cderi_oxb_mo1 = np.einsum("Puv, Avi -> AiPu", cderi, mo1_rand_half_trans)
# matmul way
nprop = mo1_rand_half_trans.shape[0]
cderi_oxb_mo1_recap = np.zeros([nprop, nocc, naux, nao])
for A in range(nprop):
    cderi_oxb_mo1_recap[A] = (mo1_rand_half_trans[A].T @ cderi.reshape(naux * nao, nao).T).reshape(nocc, naux, nao)
assert np.allclose(cderi_oxb_mo1_recap, cderi_oxb_mo1)

首先针对 `resp_bra_k0`。

$$
K_{\mu i}^\mathbb{A} \leftarrow - \widetilde{Y}_{j P \mu}^\mathbb{A} Y_{j P i}
$$

In [19]:
# einsum way
resp_bra_k0_recap = - np.einsum("AjPu, jPi -> Aui", cderi_oxb_mo1, cderi_oxo)
assert np.allclose(resp_bra_k0_recap, resp_bra_k0)
# matmul way
resp_bra_k0_recap = np.zeros([nprop, nao, nocc])
for A in range(nprop):
    resp_bra_k0_recap[A] = - cderi_oxb_mo1[A].reshape(nocc * naux, nao).T @ cderi_oxo.reshape(nocc * naux, nocc)
assert np.allclose(resp_bra_k0_recap, resp_bra_k0)

### K 积分响应 (第二部分)

我们需要两个额外的中间变量：$Y_{P i \mu}$, $\widetilde{Y}_{i P j}^\mathbb{A}$

In [20]:
cderi_xob = np.ascontiguousarray(cderi_oxb.swapaxes(0, 1))
cderi_xob.shape

(131, 5, 49)

In [21]:
# einsum way
cderi_oxo_mo1 = np.einsum("iPu, Auj -> AiPj", cderi_oxb, mo1_rand_half_trans)
# matmul way
cderi_oxo_mo1_recap = np.zeros([nprop, nocc, naux, nocc])
for A in range(nprop):
    cderi_oxo_mo1_recap[A] = (cderi_oxb.reshape(nocc * naux, nao) @ mo1_rand_half_trans[A]).reshape(nocc, naux, nocc)
assert np.allclose(cderi_oxo_mo1_recap, cderi_oxo_mo1)

其次针对 `resp_bra_k1`。

$$
K_{\mu i}^\mathbb{A} \leftarrow - \widetilde{Y}_{i P j}^\mathbb{A} Y_{P j \mu}
$$

In [22]:
# einsum way
resp_bra_k1_recap = - np.einsum("AiPj, Pju -> Aui", cderi_oxo_mo1, cderi_xob)
assert np.allclose(resp_bra_k1_recap, resp_bra_k1)
# matmul way
resp_bra_k1_recap = np.zeros([nprop, nao, nocc])
for A in range(nprop):
    resp_bra_k1_recap[A] = - cderi_xob.reshape(naux * nocc, nao).T @ cderi_oxo_mo1[A].reshape(nocc, naux * nocc).T
assert np.allclose(resp_bra_k1_recap, resp_bra_k1)

In [ ]:
# Final note: Seems for row-major, the mo1 indices should be sorted as `uPj` instead of `jPu`. We can avoid one time of explicit transposition.